# VPP Monitor — Data Setup

This notebook creates the database, schema, and tables needed by the VPP Monitor app, then loads sample data from the CSV files included in this repository.

**Prerequisites:**
- A Snowflake account with a role that can create databases (e.g., `SYSADMIN`)
- This notebook is running inside a **Snowsight Workspace** created from the Git repository (so CSV files are already available at relative paths)

**What this notebook creates:**
- Database: `EPOWER_VPP`
- Schema: `EPOWER_VPP.VPP_DATA`
- Tables: `VPP_MONITOR_TIMESERIES`, `VPP_MONITOR_ACTIONS`, `VPP_MONITOR_KPI`

## Step 1: Create Database and Schema

In [ ]:
USE ROLE SYSADMIN;

CREATE DATABASE IF NOT EXISTS EPOWER_VPP;
CREATE SCHEMA IF NOT EXISTS EPOWER_VPP.VPP_DATA;

USE SCHEMA EPOWER_VPP.VPP_DATA;

## Step 2: Create a Stage for File Upload

We create an internal stage to hold the CSV files during loading.

In [ ]:
CREATE OR REPLACE STAGE EPOWER_VPP.VPP_DATA.DATA_STAGE;

## Step 3: Upload CSV Files to the Stage

Since this notebook runs inside a Workspace created from the Git repository, the CSV files are already available at relative paths. The `PUT` command below uploads them directly to the stage.

In [ ]:
PUT file://data/vpp_monitor_timeseries.csv @EPOWER_VPP.VPP_DATA.DATA_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE;
PUT file://data/vpp_monitor_actions.csv @EPOWER_VPP.VPP_DATA.DATA_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE;
PUT file://data/vpp_monitor_kpi.csv @EPOWER_VPP.VPP_DATA.DATA_STAGE AUTO_COMPRESS=FALSE OVERWRITE=TRUE;

## Step 4: Create Tables

In [ ]:
CREATE OR REPLACE TABLE EPOWER_VPP.VPP_DATA.VPP_MONITOR_TIMESERIES (
    HOUR              TIMESTAMP_NTZ,
    REGION            VARCHAR(20),
    ACTIVE_VPP_DEVICES INTEGER,
    TOTAL_BATTERY_SOC  FLOAT,
    AVG_BATTERY_SOC_PCT FLOAT,
    TOTAL_SOLAR_YIELD_KW FLOAT,
    AVG_SOLAR_YIELD_KW FLOAT,
    NET_GRID_KW        FLOAT,
    PRICE_EUR_MWH      FLOAT,
    PRICE_EUR_KWH      FLOAT,
    DAY_OF_WEEK        VARCHAR(10),
    HOUR_OF_DAY        INTEGER
);

In [ ]:
CREATE OR REPLACE TABLE EPOWER_VPP.VPP_DATA.VPP_MONITOR_ACTIONS (
    DAY                   DATE,
    REGION                VARCHAR(20),
    CUSTOMER_TYPE         VARCHAR(50),
    BATTERY_ACTION        VARCHAR(30),
    ACTION_COUNT          INTEGER,
    TOTAL_IMPORT_KWH      FLOAT,
    TOTAL_EXPORT_KWH      FLOAT,
    TOTAL_IMPORT_COST_EUR FLOAT,
    TOTAL_EXPORT_REVENUE_EUR FLOAT,
    TOTAL_NET_MARGIN_EUR  FLOAT,
    TOTAL_CUSTOMER_MARGIN_EUR FLOAT,
    TOTAL_EPOWER_MARGIN_EUR FLOAT
);

In [ ]:
CREATE OR REPLACE TABLE EPOWER_VPP.VPP_DATA.VPP_MONITOR_KPI (
    DAY                      DATE,
    REGION                   VARCHAR(20),
    CUSTOMER_TYPE            VARCHAR(50),
    ACTIVE_VPP_DEVICES       INTEGER,
    AVG_BATTERY_SOC_PCT      FLOAT,
    AVG_SOLAR_KW             FLOAT,
    NET_GRID_KWH             FLOAT,
    AVG_PRICE_EUR_MWH        FLOAT,
    TOTAL_CUSTOMER_MARGIN_EUR FLOAT,
    TOTAL_EPOWER_MARGIN_EUR  FLOAT,
    TOTAL_NET_MARGIN_EUR     FLOAT
);

## Step 5: Load Data from Stage

In [ ]:
COPY INTO EPOWER_VPP.VPP_DATA.VPP_MONITOR_TIMESERIES
FROM @EPOWER_VPP.VPP_DATA.DATA_STAGE/vpp_monitor_timeseries.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
ON_ERROR = 'CONTINUE';

In [ ]:
COPY INTO EPOWER_VPP.VPP_DATA.VPP_MONITOR_ACTIONS
FROM @EPOWER_VPP.VPP_DATA.DATA_STAGE/vpp_monitor_actions.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
ON_ERROR = 'CONTINUE';

In [ ]:
COPY INTO EPOWER_VPP.VPP_DATA.VPP_MONITOR_KPI
FROM @EPOWER_VPP.VPP_DATA.DATA_STAGE/vpp_monitor_kpi.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
ON_ERROR = 'CONTINUE';

## Step 6: Verify Data Loaded

In [ ]:
SELECT 'VPP_MONITOR_TIMESERIES' AS table_name, COUNT(*) AS row_count FROM EPOWER_VPP.VPP_DATA.VPP_MONITOR_TIMESERIES
UNION ALL
SELECT 'VPP_MONITOR_ACTIONS', COUNT(*) FROM EPOWER_VPP.VPP_DATA.VPP_MONITOR_ACTIONS
UNION ALL
SELECT 'VPP_MONITOR_KPI', COUNT(*) FROM EPOWER_VPP.VPP_DATA.VPP_MONITOR_KPI;

## Expected Results

| Table | Expected Rows |
|-------|------|
| VPP_MONITOR_TIMESERIES | 5,760 |
| VPP_MONITOR_ACTIONS | 2,352 |
| VPP_MONITOR_KPI | 720 |

If counts match, data setup is complete. Proceed to deploy the app (see README.md).

## Cleanup (Optional)

To remove everything created by this notebook:

```sql
DROP DATABASE IF EXISTS EPOWER_VPP;
```